# TikTok Climate Action Marketing Analytics

Competition question: **How can marketing analytics help uncover insights to more effectively promote climate actions on TikTok?**

This refined notebook keeps the exploratory notebook's useful logic, but moves repeatable data preparation into `src/tiktok_semantic` and focuses the notebook on insight generation.

## Analysis Focus

1. Which post formats travel furthest?
2. Which climate message themes and action frames earn efficient engagement?
3. Which hashtags and posts point to practical creative recommendations?

The multimodal `summary.txt` files are treated as precomputed video, image, and audio understanding features.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

PROC = ROOT / "data" / "processed"
CORE = PROC / "core"
ANALYTICS = PROC / "analytics"

## Build Data

Run this once from the repo root if the processed tables are missing:

```powershell
python scripts/build_dataset.py --config configs/sample.yaml
```

In [ ]:
manifest = pd.read_csv(PROC / "manifest.csv")
display(manifest)

In [ ]:
posts = pd.read_parquet(CORE / "posts_enriched.parquet")
summaries = pd.read_parquet(CORE / "posts_summaries.parquet")
hashtags = pd.read_csv(ANALYTICS / "hashtag_performance.csv")
content_type = pd.read_csv(ANALYTICS / "content_type_performance.csv")
themes = pd.read_csv(ANALYTICS / "theme_performance.csv")
recommendations = pd.read_csv(ANALYTICS / "messaging_recommendations.csv")

print(f"Posts: {len(posts):,}")
print(f"Captured comments: {int(posts['n_comments_captured'].sum()):,}")
print(f"Video summary rows: {len(summaries):,}")

## Dataset Snapshot

In [ ]:
snapshot_cols = [
    "video_id", "date", "region", "content_type", "primary_theme", "action_frame",
    "mood_label", "play_count", "share_count", "comment_count", "eng_per_1k_views",
    "overall_narrative_message",
]
display(posts[[c for c in snapshot_cols if c in posts.columns]].sort_values("play_count", ascending=False))

## Insight 1: Format Matters

Compare video-only, image-plus-music, and image-plus-audio posts. Median views and engagement per 1k views separate reach from efficiency.

In [ ]:
display(content_type)
ax = content_type.set_index("content_type")[["median_views", "median_eng_per_1k"]].plot(kind="bar", figsize=(8, 4), secondary_y="median_eng_per_1k")
ax.set_title("Performance by TikTok Content Type")
ax.set_xlabel("")

## Insight 2: Message Frames

`primary_theme`, `action_frame`, and `mood_label` are rule-based labels derived from Gemini's multimodal summaries. They are intentionally lightweight so the analysis stays reproducible.

In [ ]:
display(themes)
display(recommendations.head(10))

## Insight 3: Hashtag Opportunities

In [ ]:
display(hashtags.head(20))
display(hashtags.query("posts >= 2").sort_values("median_eng_per_1k", ascending=False).head(15))

## Creative Recommendations Template

Use the table below as the evidence base for final presentation claims. Good claims should link a creative choice to a marketing outcome, for example: format, theme, action frame, mood, reach, shareability, or engagement efficiency.

In [ ]:
cols = ["primary_theme", "action_frame", "content_type", "posts", "median_views", "median_eng_per_1k", "median_shares", "opportunity_score"]
display(recommendations[cols])